# 🚀 HalDetect: MiniCheck SOTA Fact-Checking Benchmark (Track D)
### CSE 700 · Master Benchmark Corpus Evaluation (N = 1,005 Claims)

This notebook evaluates **MiniCheck (EMNLP 2024)** across all 1,005 claims in the HalDetect Master Benchmark Corpus using Kaggle's free GPU.

**Models supported:** `deberta-v3-large`, `flan-t5-large`, `roberta-large`, `Bespoke-MiniCheck-7B`.
**Expected Runtime:** ~60-90 seconds on GPU.

In [ ]:
# Cell 1: Install official MiniCheck from GitHub repository
!pip install -q "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"
!pip install -q transformers accelerate bitsandbytes sentencepiece scikit-learn

In [ ]:
# Cell 2: Import & Initialize MiniCheck Scorer on GPU
import os
import time
import torch
import pandas as pd
import numpy as np
from minicheck.minicheck import MiniCheck

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Model choices: 'deberta-v3-large', 'flan-t5-large', 'roberta-large', 'Bespoke-MiniCheck-7B'
MODEL_NAME = 'deberta-v3-large'
print(f"Loading MiniCheck ({MODEL_NAME})...")
t0 = time.time()
scorer = MiniCheck(model_name=MODEL_NAME, cache_dir='./ckpts')
print(f"MiniCheck loaded successfully in {time.time()-t0:.1f}s!")

In [ ]:
# Cell 3: Load Dataset & Run Fast Batch Inference
input_csv = 'kaggle_minicheck_input_claims.csv'
if not os.path.exists(input_csv):
    # Look in Kaggle input directory
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('kaggle_minicheck_input_claims.csv') or f.endswith('haldetect_master_corpus_v2.csv'):
                input_csv = os.path.join(root, f)
                break

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} claims from {input_csv}")

docs = df['evidence_passage'].fillna(df['claim']).tolist()
claims = df['claim'].tolist()

print(f"Running MiniCheck ({MODEL_NAME}) factual consistency scoring...")
t_start = time.time()

# MiniCheck score returns: pred_labels, raw_probs, _, _
pred_labels, raw_probs, _, _ = scorer.score(docs=docs, claims=claims)
elapsed = time.time() - t_start
print(f"Scored {len(claims)} claims in {elapsed:.1f}s ({len(claims)/elapsed:.1f} claims/sec)!")

# Map MiniCheck output with calibrated margin threshold
minicheck_verdicts = []
for prob in raw_probs:
    if prob >= 0.55:
        minicheck_verdicts.append('TRUE')
    elif prob <= 0.40:
        minicheck_verdicts.append('FALSE')
    else:
        minicheck_verdicts.append('UNVERIFIABLE')

df['minicheck_prob'] = raw_probs
df['minicheck_pred'] = minicheck_verdicts
df['is_correct'] = (df['minicheck_pred'] == df['gold_verdict'])
df['is_answered'] = (df['minicheck_pred'] != 'UNVERIFIABLE')

In [ ]:
# Cell 4: Compute Evaluation Metrics & Modality Breakdown
answered = df[df['is_answered']]
cov = len(answered) / len(df)
sel_acc = answered['is_correct'].mean() if len(answered) > 0 else 0.0

pred_true = df[df['minicheck_pred'] == 'TRUE']
p_true = (pred_true['gold_verdict'] == 'TRUE').mean() if len(pred_true) > 0 else 0.0

pred_false = df[df['minicheck_pred'] == 'FALSE']
p_false = (pred_false['gold_verdict'] == 'FALSE').mean() if len(pred_false) > 0 else 0.0
gold_false = df[df['gold_verdict'] == 'FALSE']
r_false = (pred_false['gold_verdict'] == 'FALSE').sum() / len(gold_false) if len(gold_false) > 0 else 0.0
f1_false = (2 * p_false * r_false / (p_false + r_false)) if (p_false + r_false) > 0 else 0.0

print("="*80)
print(f" MINICHECK ({MODEL_NAME}) EVALUATION REPORT (N={len(df)})")
print("="*80)
print(f"  • Coverage:              {cov*100:.1f}% ({len(answered)}/{len(df)})")
print(f"  • Selective Accuracy:    {sel_acc*100:.1f}% ({answered['is_correct'].sum()}/{len(answered)})")
print(f"  • Precision (TRUE):      {p_true*100:.1f}% ({len(pred_true)} predicted)")
print(f"  • Precision (FALSE):     {p_false*100:.1f}% ({len(pred_false)} predicted)")
print(f"  • Hallucination F1:      {f1_false:.3f} (Recall: {r_false*100:.1f}%)")
print("-"*80)

print(f"\n{'Modality':<20} | {'Total':<6} | {'Cov %':<6} | {'Sel Acc %':<10} | {'P(TRUE)':<8} | {'P(FALSE)':<8} | {'F1(FALSE)'}")
print(f"{'-'*20}-+-{'-'*6}-+-{'-'*6}-+-{'-'*10}-+-{'-'*8}-+-{'-'*8}-+-{'-'*10}")
for mod, g in df.groupby('type'):
    ans_g = g[g['is_answered']]
    m_cov = len(ans_g) / len(g)
    m_acc = ans_g['is_correct'].mean() if len(ans_g) > 0 else 0.0
    pt = (g[g['minicheck_pred'] == 'TRUE']['gold_verdict'] == 'TRUE').mean() if (g['minicheck_pred'] == 'TRUE').sum() > 0 else 0.0
    pf = (g[g['minicheck_pred'] == 'FALSE']['gold_verdict'] == 'FALSE').mean() if (g['minicheck_pred'] == 'FALSE').sum() > 0 else 0.0
    gf = (g['gold_verdict'] == 'FALSE').sum()
    rf = (g[g['minicheck_pred'] == 'FALSE']['gold_verdict'] == 'FALSE').sum() / gf if gf > 0 else 0.0
    f1f = (2*pf*rf/(pf+rf)) if (pf+rf) > 0 else 0.0
    print(f"{mod:<20} | {len(g):<6} | {m_cov*100:>5.1f}% | {m_acc*100:>9.1f}% | {pt*100:>6.1f}% | {pf*100:>6.1f}% | {f1f:>8.3f}")
print("="*80)

In [ ]:
# Cell 5: Save Output CSV for HalDetect Integration
out_csv = 'haldetect_minicheck_results.csv'
df.to_csv(out_csv, index=False)
print(f"Saved complete rescored results to {out_csv}!")